# Section 1: Setup & Data Prep

In [ ]:
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

# Load the master dataset
df = pd.read_csv('master_dataset.csv')
df['clean_text'] = df['clean_text'].fillna('')

# Create Combined_Text column (fallback in case it differs, here we use clean_text)
df['Combined_Text'] = df['clean_text']

# Define the F1-specific stop words
f1_noise_words = [
    'circuit', 'Circuit', 'specific', 'Specific', 'brought', 'upgrade', 'team', 'car', 'event', 
    'geometry', 'changes', 'description', 'introduced', 'previous', 'version', 'suit', 'allow', 
    'appropriate', 'balance', 'achieved', 'track', 'race', 'performance', 'Performance', 
    'modified', 'modifier', 'order', 'range', 'combination', 'efficiently', 'reducing', 
    'suitable', 'high', 'low', 'new', 'development', 'rear', 'Rear', 'front', 'Front', 
    'wing', 'Wing', 'local', 'Local', 'load', 'Load', 'doc', 'submissions', 'presentation',
    'improved', 'Improved', 'lower', 'Lower', 'efficiency', 'Efficiency', 'revised', 'Revised', 
    'update', 'Update', 'updates', 'order in', 'aimed', 'aimed at', 'brought a', 'results'
]
custom_stop_words = list(ENGLISH_STOP_WORDS) + f1_noise_words

# Section 2: Unsupervised Clustering (PCA + K-Means)

In [ ]:
# 1. Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(
    max_df=0.95, 
    min_df=2, 
    stop_words=custom_stop_words, 
    ngram_range=(1, 2), 
    max_features=500,
    lowercase=True
)
X_tfidf = vectorizer.fit_transform(df['Combined_Text'])

# 2. Apply TruncatedSVD for PCA (Dimensionality Reduction to 2 components)
svd = TruncatedSVD(n_components=2, random_state=42)
pca_components = svd.fit_transform(X_tfidf)
df['PCA_X'] = pca_components[:, 0]
df['PCA_Y'] = pca_components[:, 1]

# 3. Apply KMeans Clustering
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_tfidf)

# Map clusters to categories
cluster_names = {
    0: "Cooling & Reliability",
    1: "Floor & Underbody",
    2: "Circuit-Specific Aerodynamics",
    3: "General Aerodynamic Performance",
    4: "Bodywork Flow Conditioning"
}
df['Engineering_Category'] = [cluster_names[c] for c in clusters]

# Section 3: Interactive Visualizations (Plotly)

In [ ]:
# 1. Interactive Scatter Plot (PCA)
fig_scatter = px.scatter(
    df, x='PCA_X', y='PCA_Y', color='Engineering_Category', hover_data=['team', 'Race'],
    title='PCA Clustering of F1 Technical Upgrades by Engineering Category'
)
fig_scatter.show()

# 2. Team Upgrades Bar Chart
team_counts = df['team'].value_counts().reset_index()
team_counts.columns = ['Team', 'Upgrades']
fig_bar = px.bar(
    team_counts, x='Team', y='Upgrades', color='Team',
    title='Total Upgrades Brought by Each Team'
)
fig_bar.show()

# 3. Upgrade Breakdown Treemap
fig_tree = px.treemap(
    df, path=['team', 'Engineering_Category'], 
    title='Treemap of Upgrade Categories by Team'
)
fig_tree.show()

# 4. Chronological Histogram (Upgrades per Race)
fig_hist = px.histogram(
    df, x='Race', color='team', 
    title='Chronological Timeline of Upgrades Brought per Race',
    category_orders={'Race': df['Race'].unique().tolist()}
)
fig_hist.show()

# Section 4: Supervised ML & Feature Importance

In [ ]:
# 1. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, df['is_successful'], test_size=0.2, random_state=42, stratify=df['is_successful']
)

# 2. Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 3. Print Classification Report
y_pred = rf_model.predict(X_test)
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

# 4. Plot Top 15 Feature Importance Bar Chart using Seaborn
importances = rf_model.feature_importances_
feature_names = vectorizer.get_feature_names_out()
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
top_15_features = feature_importance_df.sort_values(by='Importance', ascending=False).head(15)

plt.figure(figsize=(10, 8))
sns.barplot(data=top_15_features, x='Importance', y='Feature', palette='magma')
plt.title('Top 15 Technical Feature Weights Driving Team Performance Gains', fontsize=16)
plt.xlabel('Random Forest Gini Importance Score', fontsize=12)
plt.ylabel('Engineering Component / Keyword', fontsize=12)
plt.tight_layout()
plt.show()

# Section 5: Information Retrieval (The Search Engine)

In [ ]:
def search_upgrades(query, vectorizer, tfidf_matrix, df, top_n=3):
    """
    Vectorizes a user query and returns the top matching upgrades based on cosine similarity.
    """
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[-top_n:][::-1]
    
    print(f"🔍 Search Results for: '{query}'\n" + "-"*50)
    for rank, idx in enumerate(top_indices, 1):
        sim_score = similarities[idx]
        team = df.iloc[idx]['team']
        date = df.iloc[idx]['date']
        category = df.iloc[idx]['Engineering_Category']
        snippet = str(df.iloc[idx]['Combined_Text'])[:150] + "..."
        
        print(f"{rank}. {team} ({date}) | Category: {category} | Sim Score: {sim_score:.3f}")
        print(f"   Snippet: {snippet}\n")

# Example Search
search_upgrades("high speed downforce floor edge", vectorizer, X_tfidf, df, top_n=3)